In [1]:
import os
import pandas as pd
import numpy as np
from utils import decomposition
import importlib
importlib.reload(decomposition)

<module 'utils.decomposition' from '/Users/tony/Documents/sisepuede_modeling/ssp_louisiana/1000_runs_ensamble_postprocessing/utils/decomposition.py'>

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
td = decomposition.TemporalDecomposition()

In [4]:
# Set up paths
SCRIPT_DIR_PATH = os.getcwd()
CW_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "cw")
DATA_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "data")
OUTPUT_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "output")

In [5]:
# Load emissions targets
te_all = pd.read_csv(os.path.join(CW_DIR_PATH, "emission_targets_louisiana.csv"))
target_country = "LA"
cols_needed = ["Subsector", "Gas", "Vars", "Edgar_Class", target_country]
te_all = te_all[cols_needed].copy()
te_all["tvalue"] = te_all[target_country]
te_all = te_all.drop(columns=[target_country])
te_all

,Subsector,Gas,Vars,Edgar_Class,tvalue
0,lvst,ch4,emission_co2e_ch4_lvst_entferm_buffalo:emissio...,AG - Livestock:CH4,1.539811
1,lsmm,ch4,emission_co2e_ch4_lsmm_anaerobic_digester:emis...,AG - Livestock:CH4,0.151107
2,lsmm,n2o,emission_co2e_n2o_lsmm_direct_anaerobic_digest...,AG - Livestock:N2O,0.079212
3,agrc,co2,emission_co2e_co2_agrc_biomass_bevs_and_spices...,AG - Crops:CO2,0.000000
4,agrc,ch4,emission_co2e_ch4_agrc_anaerobicdom_rice:emiss...,AG - Crops:CH4,2.384974
...,...,...,...,...,...
63,soil,co2,emission_co2e_co2_soil_lime_use:emission_co2e_...,LULUCF - Organic Soil:CO2,0.305152
64,soil,n2o,emission_co2e_n2o_soil_fertilizer:emission_co2...,LULUCF - Organic Soil:N2O,0.928029
65,ccsq,ch4,emission_co2e_ch4_ccsq_direct_air_capture,CCSQ:CH4,0.000000
66,ccsq,co2,emission_co2e_co2_ccsq_direct_air_capture,CCSQ:CO2,0.000000


In [6]:
# Parse target variables
te_all["Vars_list"] = te_all["Vars"].str.split(":")
target_vars = [item for sublist in te_all["Vars_list"].tolist() for item in sublist]
print("Target variables:", target_vars[:10])  # Display first 10 target variables
print("Total target variables:", len(target_vars))

Target variables: ['emission_co2e_ch4_lvst_entferm_buffalo', 'emission_co2e_ch4_lvst_entferm_cattle_dairy', 'emission_co2e_ch4_lvst_entferm_cattle_nondairy', 'emission_co2e_ch4_lvst_entferm_chickens', 'emission_co2e_ch4_lvst_entferm_goats', 'emission_co2e_ch4_lvst_entferm_horses', 'emission_co2e_ch4_lvst_entferm_mules', 'emission_co2e_ch4_lvst_entferm_pigs', 'emission_co2e_ch4_lvst_entferm_sheep', 'emission_co2e_ch4_lsmm_anaerobic_digester']
Total target variables: 470


In [9]:
# Output folders
ensemble_id = "de38bb46-7d00-4cf5-8844-f6cc20695024"
PARSED_RUNS_DIR_PATH = os.path.join(DATA_DIR_PATH, "parsed_runs")
ENSEMBLE_PARSED_DIR_PATH = os.path.join(PARSED_RUNS_DIR_PATH, ensemble_id)
files_names = [f for f in os.listdir(ENSEMBLE_PARSED_DIR_PATH) if f.endswith('.csv')]

In [10]:
len(files_names)

1001

In [11]:
PARSED_RUNS_PROCESSED_DIR_PATH = os.path.join(DATA_DIR_PATH, "parsed_runs_processed")
ENSEMBLE_PARSED_PROCESSED_DIR_PATH = os.path.join(PARSED_RUNS_PROCESSED_DIR_PATH, ensemble_id)
os.makedirs(PARSED_RUNS_PROCESSED_DIR_PATH, exist_ok=True)
os.makedirs(ENSEMBLE_PARSED_PROCESSED_DIR_PATH, exist_ok=True)

In [12]:
time_period_ref = 7

# --- process each batch file with your updated rescale() ---
for run, output_file in enumerate(files_names):
    path_in = os.path.join(ENSEMBLE_PARSED_DIR_PATH, output_file)
    df_in = pd.read_csv(path_in)

    # keep only years >= t0
    df_in = df_in[df_in["time_period"] >= time_period_ref].copy()
    if df_in.empty:
        print(f"[skip] {output_file} has no rows >= t0")
        continue

    # single region assumption (but still grab it from data)
    region = df_in["region"].iloc[0]

    # choose a baseline id that exists in THIS batch
    ref_id = 0 if (df_in["primary_id"] == 0).any() else int(df_in["primary_id"].min())

    # run rescale; rescale writes its own output file
    td.rescale(
        z=0,
        rall=np.array([region]),
        data_all=df_in,
        te_all=te_all,
        initial_conditions_id=[ref_id],
        dir_output=ENSEMBLE_PARSED_PROCESSED_DIR_PATH,
        time_period_ref=time_period_ref,
        run=run
    )
    print(f"[done] run={run} file={output_file} region={region} baseline_id={ref_id}")


Saved: /Users/tony/Documents/sisepuede_modeling/ssp_louisiana/1000_runs_ensamble_postprocessing/data/parsed_runs_processed/de38bb46-7d00-4cf5-8844-f6cc20695024/louisiana_0.csv
[done] run=0 file=545.csv region=louisiana baseline_id=354898
Saved: /Users/tony/Documents/sisepuede_modeling/ssp_louisiana/1000_runs_ensamble_postprocessing/data/parsed_runs_processed/de38bb46-7d00-4cf5-8844-f6cc20695024/louisiana_1.csv
[done] run=1 file=223.csv region=louisiana baseline_id=354576
Saved: /Users/tony/Documents/sisepuede_modeling/ssp_louisiana/1000_runs_ensamble_postprocessing/data/parsed_runs_processed/de38bb46-7d00-4cf5-8844-f6cc20695024/louisiana_2.csv
[done] run=2 file=237.csv region=louisiana baseline_id=354590
Saved: /Users/tony/Documents/sisepuede_modeling/ssp_louisiana/1000_runs_ensamble_postprocessing/data/parsed_runs_processed/de38bb46-7d00-4cf5-8844-f6cc20695024/louisiana_3.csv
[done] run=3 file=551.csv region=louisiana baseline_id=354904
Saved: /Users/tony/Documents/sisepuede_modeling/

In [13]:
# collect decomposed runs
files_out = [f for f in os.listdir(ENSEMBLE_PARSED_PROCESSED_DIR_PATH) if f.endswith(".csv")]
data_complete = pd.concat(
    [pd.read_csv(os.path.join(ENSEMBLE_PARSED_PROCESSED_DIR_PATH, f)) for f in files_out],
    ignore_index=True
)

# (optional) dedupe exact duplicates
key_cols = ["region", "primary_id", "time_period"]
data_complete = data_complete.drop_duplicates(subset=key_cols + [c for c in data_complete.columns if c not in key_cols])

In [14]:
data_complete.head()

,Index,time_period,primary_id,region,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,...,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_waso,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_ippu
0,louisiana_354660,7,354660,louisiana,0.0,356696.043492,66146.621297,77.773211,76769.151304,6508.599365,...,1.233181,3.138402,0.447204,0.0,30.469259,13.094104,117.042622,4.491483,45.130223,2.629950
1,louisiana_354660,8,354660,louisiana,0.0,355221.860914,65873.245131,77.451784,76451.873477,6481.700093,...,1.227027,3.190377,0.453932,0.0,38.785521,15.448650,114.205980,4.519073,46.068411,2.641921
2,louisiana_354660,9,354660,louisiana,0.0,353750.075712,65600.313541,77.130879,76135.111621,6454.844566,...,1.210373,3.241353,0.460975,0.0,35.280608,14.868080,114.046420,4.549158,47.114101,2.656241
3,louisiana_354660,10,354660,louisiana,0.0,352280.764654,65327.840762,76.810514,75818.882257,6428.034184,...,1.181755,3.293836,0.468262,0.0,47.213230,17.172010,112.604235,4.581895,48.249651,2.672838
4,louisiana_354660,11,354660,louisiana,0.0,350814.002751,65055.840705,76.490704,75503.201530,6401.270317,...,1.140940,3.347158,0.475743,0.0,47.401636,17.516973,111.117115,4.617352,49.461277,2.691630


In [15]:
time_period_ref

7

In [16]:
# --- GLOBAL t0 EQUALIZATION (Option A) ---
# mapped_only=False => equalize all co2e_ vars; set True to equalize only those in te_all.Vars_list
data_complete = td.enforce_global_t0_equalization(
    data_complete,
    time_period_ref=time_period_ref,
    te_all=te_all,
    mapped_only=False
)

data_complete = td.recompute_subsector_totals(data_complete, te_all)

# --- VALIDATE ---
print(td.assert_equal_t0(data_complete, 7))           # base vars
print(td.assert_equal_t0_totals(data_complete, 7))    # subsector totals


True
True


In [21]:
# Filter some outlier runds
# primary_ids_to_remove = [354920, 355090, 354624]
primary_ids_to_remove = [354790, 354833, 355220]
data_complete = data_complete[~data_complete["primary_id"].isin(primary_ids_to_remove)]
data_complete.shape

(28855, 3976)

In [18]:
# check for fields full of nans
data_complete.isnull().sum()

Index                                 0
time_period                           0
primary_id                            0
region                                0
area_agrc_crops_bevs_and_spices       0
                                     ..
emission_co2e_subsector_total_fgtv    0
emission_co2e_subsector_total_inen    0
emission_co2e_subsector_total_scoe    0
emission_co2e_subsector_total_trns    0
emission_co2e_subsector_total_ippu    0
Length: 3976, dtype: int64

In [19]:
data_complete.info()

<class 'pandas.core.frame.DataFrame'>
Index: 28942 entries, 0 to 29028
Columns: 3976 entries, Index to emission_co2e_subsector_total_ippu
dtypes: float64(3972), int64(2), object(2)
memory usage: 878.2+ MB


In [22]:
# --- WRITE FINAL OUTPUT ---
final_name = f"sisepuede_results_IDE_{ensemble_id}.csv"
out_path = os.path.join(OUTPUT_DIR_PATH, final_name)
data_complete.to_csv(out_path, index=False)
print(f"[final] wrote {out_path}")

[final] wrote /Users/tony/Documents/sisepuede_modeling/ssp_louisiana/1000_runs_ensamble_postprocessing/output/sisepuede_results_IDE_de38bb46-7d00-4cf5-8844-f6cc20695024.csv
